In [1]:
from dotenv import load_dotenv
from mlModels.regression.data.data import getData, getRegressionData
from mlModels.kmeans.locationClustering import findBestFittingK
from database.db import getConnection

import pandas as pd
import os

CV = True
CV_VAL = 5

os.chdir("/home/florian/Desktop/immopreis-regression")

load_dotenv("database/.env")
conn = getConnection()

df_listings = pd.read_sql("SELECT * FROM listings", conn)
df_features = pd.read_sql("SELECT * FROM features", conn)

df_features = getData(filter_type="finance_type", filter_val="rent")
df = pd.read_sql("""
                 SELECT l.id, l.lat, l.lon
                 FROM listings l
                 """, conn)
conn.close()

cluster_dict = findBestFittingK(df, seed=42, lower=2, upper=25)

# XGBoost

In [2]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor as xgbr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df_features_run = df_features.copy()
results = []

for k, df_cluster in cluster_dict.items():

    df = df_features_run.merge(df_cluster, how="inner", on="id")

    df_house_X, df_house_y, df_apt_X, df_apt_y = getRegressionData(df)
    print(df_apt_X.corrwith(df_apt_y).sort_values())


    X_train, X_test, y_train, y_test = train_test_split(df_apt_X, df_apt_y, test_size=0.2, random_state=42)

    drop_cols = []

    X_train.drop(columns=drop_cols, inplace=True, errors="ignore")
    X_test.drop(columns=drop_cols, inplace=True, errors="ignore")

    model = xgbr(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(model, X_train, y_train, cv=CV_VAL, scoring="r2", verbose=True)
    print(f"\n-------Cross Validation Scores for K={k}-------")
    print(f"CV R2 Scores: {scores}")
    print(f"Mean R2 Score: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print()

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"-------XGBoost Regression Results for K={k}-------")
    print(f"MAE:  {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2:   {r2:.4f}")
    importance_df = pd.DataFrame({
        "feature": X_train.columns,
        "importance": model.feature_importances_
    })
    results.append({
        "k": k,
        "Mean R2": scores.mean(),
        "STD": scores.std(),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })
    importance_df = importance_df.sort_values("importance", ascending=False)
    print(importance_df.tail(20))
    print()

ks = [r["k"] for r in results]
mean_R2s = [r["Mean R2"] for r in results]
stds = [r["STD"] for r in results]
maes = [r["MAE"] for r in results]
rmses = [r["RMSE"] for r in results]
R2s = [r["R2"] for r in results]

plt.figure()
plt.plot(ks, mean_R2s, label="Mean R2 CV=5", color="blue")
plt.plot(ks, stds, label="Standard Deviation", color="red")
plt.plot(ks, maes, label="MAE", color="green")
plt.plot(ks, rmses, label="RMSE", color="yellow")
plt.plot(ks, R2s, label="R2", color="black")
plt.xlabel("K")
plt.legend()
plt.grid()
plt.show()


KeyboardInterrupt: 